Install dependencies

In [ ]:
%pip install -q tokenizers datasets

Load Train and validation splits

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "Salesforce/wikitext",
    "wikitext-2-raw-v1"
)

train_data = dataset["train"]
validation_data = dataset["validation"]

print("Training rows:", len(train_data))
print("Validation rows:", len(validation_data))
print("Sample:", train_data[0]["text"][:300])

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…): reconstructing file:   0%|          |  0.00B /  733kB            

wikitext-2-raw-v1/test-00000-of-00001.pa(…): downloading bytes:           |  0.00B            

wikitext-2-raw-v1/train-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 6.36MB            

wikitext-2-raw-v1/train-00000-of-00001.p(…): downloading bytes:           |  0.00B            

...

wikitext-2-raw-v1/validation-00000-of-00(…): downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

Training rows: 36718
Validation rows: 3760
Sample: 


Define a Text iterator

In [ ]:
def text_iterator(split, max_rows=None):
    count = 0
    for row in split:
        text = row["text"]
        if text.strip():
            yield text
            count += 1
            if max_rows is not None and count >= max_rows:
                break

Create and train byte-level BPE

In [ ]:
from tokenizers import Tokenizer, models, trainers
from tokenizers import pre_tokenizers, decoders, normalizers

VOCAB_SIZE = 8000  # Change to 2000 for a smaller experiment

tokenizer = Tokenizer(
    models.BPE(unk_token="<|unk|>")
)

tokenizer.normalizer = normalizers.NFC()
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(
    add_prefix_space=False
)
tokenizer.decoder = decoders.ByteLevel()

trainer = trainers.BpeTrainer(
    vocab_size=VOCAB_SIZE,
    min_frequency=2,
    special_tokens=["<|unk|>"],
    initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
    show_progress=True
)

tokenizer.train_from_iterator(
    text_iterator(train_data),
    trainer=trainer
)

print("Actual vocabulary size:", tokenizer.get_vocab_size())

Actual vocabulary size: 8000


Encode and decode a test sentence

In [ ]:
sample = (
    "Mechanical engineers use simulation to study forces, "
    "stress, fluid motion, and vehicle aerodynamics."
)

encoded = tokenizer.encode(sample)

print("Original:", sample)
print("Tokens:", encoded.tokens)
print("Token IDs:", encoded.ids)
print("Token count:", len(encoded.ids))
print("Decoded:", tokenizer.decode(encoded.ids))

Original: Mechanical engineers use simulation to study forces, stress, fluid motion, and vehicle aerodynamics.
Tokens: ['M', 'echan', 'ical', 'Ġengine', 'ers', 'Ġuse', 'Ġsim', 'ulation', 'Ġto', 'Ġstudy', 'Ġforces', ',', 'Ġst', 'ress', ',', 'Ġflu', 'id', 'Ġmotion', ',', 'Ġand', 'Ġvehic', 'le', 'Ġaer', 'ody', 'n', 'am', 'ics', '.']
Token IDs: [45, 4411, 515, 2620, 383, 1148, 1139, 1451, 290, 2277, 1817, 12, 348, 629, 12, 5861, 334, 6164, 12, 288, 4277, 297, 5736, 1333, 78, 314, 853, 14]
Token count: 28
Decoded: Mechanical engineers use simulation to study forces, stress, fluid motion, and vehicle aerodynamics.


Inspect vocabulary entries

In [ ]:
vocab = tokenizer.get_vocab()
print("Vocabulary entries:", len(vocab))

for token, token_id in sorted(vocab.items(), key=lambda item: item[1])[:50]:
    print(f"{token_id:5d}  {token!r}")

Vocabulary entries: 8000
    0  '<|unk|>'
    1  '!'
    2  '"'
    3  '#'
    4  '$'
    5  '%'
    6  '&'
    7  "'"
    8  '('
    9  ')'
   10  '*'
   11  '+'
   12  ','
   13  '-'
   14  '.'
   15  '/'
   16  '0'
   17  '1'
   18  '2'
   19  '3'
   20  '4'
   21  '5'
   22  '6'
   23  '7'
   24  '8'
   25  '9'
   26  ':'
   27  ';'
   28  '<'
   29  '='
   30  '>'
   31  '?'
   32  '@'
   33  'A'
   34  'B'
   35  'C'
   36  'D'
   37  'E'
   38  'F'
   39  'G'
   40  'H'
   41  'I'
   42  'J'
   43  'K'
   44  'L'
   45  'M'
   46  'N'
   47  'O'
   48  'P'
   49  'Q'


Cell 7 --- Inspect learned BPE merge rules

In [ ]:
import json

model_data = json.loads(tokenizer.to_str())
merges = model_data["model"].get("merges", [])

print("Merge rules:", len(merges))
print("First 20 merge rules:")
for rule in merges[:20]:
    print(rule)

Merge rules: 7743
First 20 merge rules:
['Ġ', 't']
['h', 'e']
['Ġ', 'a']
['i', 'n']
['Ġt', 'he']
['e', 'r']
['o', 'n']
['Ġ', ',']
['r', 'e']
['Ġ', 's']
['e', 'd']
['Ġ', 'o']
['Ġ', 'w']
['n', 'd']
['a', 't']
['Ġ', '.']
['o', 'r']
['i', 't']
['Ġ', 'c']
['e', 'n']


 Evaluate on held-out validation text

In [ ]:
def evaluate(split, max_rows=1000):
    total_bytes = 0
    total_tokens = 0
    examples = 0

    for text in text_iterator(split, max_rows=max_rows):
        total_bytes += len(text.encode("utf-8"))
        total_tokens += len(tokenizer.encode(text).ids)
        examples += 1

    print("Examples:", examples)
    print("UTF-8 bytes:", total_bytes)
    print("Token IDs:", total_tokens)
    print("Bytes per token:", round(total_bytes / max(total_tokens, 1), 3))

evaluate(validation_data)

Examples: 1000
UTF-8 bytes: 425171
Token IDs: 108285
Bytes per token: 3.926


 Download the tokenizer to your computer


In [ ]:
# Save the tokenizer to disk first
tokenizer.save("my_bpe_tokenizer.json")

# Download the saved file to your computer
from google.colab import files
files.download("my_bpe_tokenizer.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Reload and verify

In [ ]:
from tokenizers import Tokenizer

loaded_tokenizer = Tokenizer.from_file("my_bpe_tokenizer.json")

test_text = "A Transformer predicts the next token."
result = loaded_tokenizer.encode(test_text)

print("Tokens:", result.tokens)
print("IDs:", result.ids)
print("Decoded:", loaded_tokenizer.decode(result.ids))

Tokens: ['A', 'ĠTrans', 'form', 'er', 'Ġpred', 'ict', 's', 'Ġthe', 'Ġnext', 'Ġto', 'ken', '.']
IDs: [33, 3807, 713, 262, 2154, 703, 83, 261, 1596, 290, 3626, 14]
Decoded: A Transformer predicts the next token.


In [ ]:
vocab = tokenizer.get_vocab()

print("Actual vocabulary size:", tokenizer.get_vocab_size())
print("First 20 vocabulary entries:")

for token, token_id in sorted(
    vocab.items(), key=lambda item: item[1]
)[:8000]:
    print(f"{token_id}: {token!r}")

Actual vocabulary size: 8000
First 20 vocabulary entries:
0: '<|unk|>'
1: '!'
2: '"'
3: '#'
4: '$'
5: '%'
6: '&'
7: "'"
8: '('
9: ')'
10: '*'
11: '+'
12: ','
13: '-'
14: '.'
15: '/'
16: '0'
17: '1'
18: '2'
19: '3'
20: '4'
21: '5'
22: '6'
23: '7'
24: '8'
25: '9'
26: ':'
27: ';'
28: '<'
29: '='
30: '>'
31: '?'
32: '@'
33: 'A'
34: 'B'
35: 'C'
36: 'D'
37: 'E'
38: 'F'
39: 'G'
40: 'H'
41: 'I'
42: 'J'
43: 'K'
44: 'L'
45: 'M'
46: 'N'
47: 'O'
48: 'P'
49: 'Q'
50: 'R'
51: 'S'
52: 'T'
53: 'U'
54: 'V'
55: 'W'
56: 'X'
57: 'Y'
58: 'Z'
59: '['
60: '\\'
61: ']'
62: '^'
63: '_'
64: '`'
65: 'a'
66: 'b'
67: 'c'
68: 'd'
69: 'e'
70: 'f'
71: 'g'
72: 'h'
73: 'i'
74: 'j'
75: 'k'
76: 'l'
77: 'm'
78: 'n'
79: 'o'
80: 'p'
81: 'q'
82: 'r'
83: 's'
84: 't'
85: 'u'
86: 'v'
87: 'w'
88: 'x'
89: 'y'
90: 'z'
91: '{'
92: '|'
93: '}'
94: '~'
95: '¡'
96: '¢'
97: '£'
98: '¤'
99: '¥'
100: '¦'
101: '§'
102: '¨'
103: '©'
104: 'ª'
105: '«'
106: '¬'
107: '®'
108: '¯'
109: '°'
110: '±'
111: '²'
112: '³'
113: '´'
114: 'µ'
115: '¶'
1

In [ ]:
text = "the farmer goes to this home daily"

encoded = tokenizer.encode(text)

print("Tokens:", encoded.tokens)
print("Token IDs:", encoded.ids)
print("Token count:", len(encoded.ids))
print("Decoded text:", tokenizer.decode(encoded.ids))

Tokens: ['t', 'he', 'Ġfar', 'mer', 'Ġgoes', 'Ġto', 'Ġthis', 'Ġhome', 'Ġdaily']
Token IDs: [84, 258, 2359, 558, 6147, 290, 589, 1461, 5707]
Token count: 9
Decoded text: the farmer goes to this home daily


In [ ]:
import torch
import torch.nn as nn

vocab_size = tokenizer.get_vocab_size()
embedding_dim = 128

embedding = nn.Embedding(embedding_dim,vocab_size)

torch.save(
    embedding.weight.detach().cpu(),
    "embedding_matrix.pt"
)

print("Shape:", tuple(embedding.weight.shape))

Shape: (128, 8000)


In [ ]:
from google.colab import files
files.download("embedding_matrix.pt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>